# 2. Cleaning and validation

Cleaning follows evidence. Missing Ladakh 2020 stays null; the Jammu and Kashmir 2020 footnote stays documented. We validate duplicates and impossible negative counts without automatically dropping rows.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import data_collection, data_cleaning
data_collection.main()
national, states = data_cleaning.clean_india()
who = data_cleaning.clean_global()
national.head(), states.isna().sum(), who.shape

`groupby` splits rows into groups, applies a summary, and combines results. Here it can reconcile state totals by year against the published national series; group totals are a data-quality check, not evidence that the measures are comparable across sources.

In [ ]:
state_totals = states.groupby('year', dropna=False)['accidents'].sum(min_count=1)
pd.concat([national.set_index('year')['accidents'].rename('national'), state_totals.rename('state_sum')], axis=1).assign(difference=lambda x: x.state_sum - x.national)